<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/09_GES_Aware_Genomic_RAG_Cell_7C2_Quality_Reranking_and_Top5_Execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder name is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen identities, paths, and overwrite protection

In [ ]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import math
import os
import re
import shutil
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = '09_GES_Aware_Genomic_RAG_Cell_7C2_Quality_Reranking_and_Top5_Execution.ipynb'
CELL_ID = '7C2'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_QUESTIONS = 80
EXPECTED_TOP20_PER_QUESTION = 20
EXPECTED_TOP5_PER_QUESTION_CONDITION = 5
EXPECTED_CONDITIONS = 6
EXPECTED_TOP20_ROWS = 1_600
EXPECTED_AUDIT_ROWS = 9_600
EXPECTED_FINAL_TOP5_ROWS = 2_400
FROZEN_RANDOM_SEED = 20260722

EXPECTED_CELL_7C1_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C2_FROZEN_CELL7A3_SCORE_LOADING_SIX_CONDITION_QUALITY_RANKING_'
    'FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_AND_FINAL_TOP5_MATERIALIZATION_ONLY_'
    'NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

EXPECTED_CELL_7C1_TERMINAL_DECISION = (
    'PASS_STAGE7C1_COMPLETE_CELL7C0_PACKAGE_CELL7A3_SCORE_PACKAGE_AND_CELL7B4_QUALITY_'
    'CONFIGURATION_REVERIFIED_CHECKSUM_PROTECTED_CELL7C2_SCORE_LOADING_SIX_CONDITION_'
    'QUALITY_RANKING_FIXED_RRF_AND_FINAL_TOP5_MATERIALIZATION_ONLY_AUTHORIZED_NO_SCORES_'
    'OPENED_IN_AUTHORIZATION_NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_'
    'ADJUDICATION_OR_RAG_METRICS'
)

EXPECTED_CELL_7C0_DECISION = (
    'PASS_STAGE7C0_EXACT_PUBMEDBERT_REVISION_CORPUS_AND_QUESTION_EMBEDDINGS_FLOAT32_L2_768_'
    'EXACT_FAISS_INDEXFLATIP_COMMON_SEMANTIC_TOP20_80_QUESTIONS_1600_CANDIDATES_MATERIALIZED_'
    'CHECKSUM_PROTECTED_NO_CELL7A3_SCORES_QUALITY_RERANKING_RRF_TOP5_PROMPTS_LLM_ANSWER_KEY_'
    'OUTCOME_INSPECTION_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7A3_DECISION = (
    'PASS_STAGE7A3_FROZEN_T1_FULL_GES_NO_STAR_GES_AND_COMBINED_METADATA_SCORES_MATERIALIZED_'
    'CHECKSUM_PROTECTED_T0_PREDICT_PROBA_REPRODUCED_NO_FITTING_NO_THRESHOLDING_NO_RANKING_'
    'NO_RAG_CORPUS_EMBEDDINGS_OR_LLM_NEXT_STAGE_REQUIRES_SEPARATE_PROTOCOL_FREEZE'
)

EXPECTED_CELL_7B4_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_SIMILARITY_TOP20_'
    'CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_BLINDED_ALIASES_PROMPTS_STRICT_'
    'RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_'
    'FROZEN_CHECKSUM_PROTECTED_NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_'
    'LLM_ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------------------
# Cell 7C1 V2 authorization package
# --------------------------------------------------------------------------------------------------
CELL_7C1_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c1_quality_reranking_authorization_v1'
)
CELL_7C1_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c1_quality_reranking_authorization_v1'
)

CELL_7C1 = OrderedDict([
    ('authorization', {
        'path': CELL_7C1_CONFIG_DIR / 'cell_7c1_stage7c_cell7c2_quality_reranking_top5_execution_authorization_v1.json',
        'sha256': 'd9bab8e4f9aa7cc015d21647e09cb87ba916fd873f038967637289addc633336',
    }),
    ('input_inventory', {
        'path': CELL_7C1_CONFIG_DIR / 'cell_7c1_authorized_quality_reranking_input_inventory_v1.csv',
        'sha256': '1f214d36062d8d52af953e07e65545d2b36751e3a9d18614e9f29a44e9944af3',
    }),
    ('qc', {
        'path': CELL_7C1_QC_DIR / 'cell_7c1_quality_reranking_authorization_qc_v1.json',
        'sha256': '7233279fcc5f8ab180c1fdba707a004880664033cfb9333ea5201cd75f1f5f4c',
    }),
    ('manifest', {
        'path': CELL_7C1_CONFIG_DIR / 'cell_7c1_quality_reranking_authorization_manifest_v1.json',
        'sha256': 'f5a31f64e47e60087365ee58e75880354c48f76ef7fc941e5840aa1ec7360466',
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact frozen Cell 7C0 package
# --------------------------------------------------------------------------------------------------
CELL_7C0_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)
CELL_7C0_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)
CELL_7C0_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)

CELL_7C0 = OrderedDict([
    ('model_artifact_inventory', {
        'path': CELL_7C0_DIR / 'cell_7c0_embedding_model_artifact_inventory_v1.json',
        'sha256': '1bdca6adcfbb31eb81d928bf103c5821d42e93695653debcec6041c442064889',
    }),
    ('corpus_embeddings', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_embeddings_float32_l2_v1.npy',
        'sha256': '3f9360f39c8730131e0784d82cb6a72a4dc14b37e4e177cc0368cfbffed1df9a',
    }),
    ('corpus_embedding_identity', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_embedding_identity_v1.parquet',
        'sha256': '8c853afc5f6db4d54dc9d73d1b24dbece5e13bbfa891db978f6f5ec42782153e',
    }),
    ('question_embeddings', {
        'path': CELL_7C0_DIR / 'cell_7c0_primary_question_embeddings_float32_l2_v1.npy',
        'sha256': '163394b376eda504d516ad6d74c01d79f48a247785aa909a6e3b1f8d7be36ff2',
    }),
    ('question_embedding_identity', {
        'path': CELL_7C0_DIR / 'cell_7c0_primary_question_embedding_identity_v1.csv',
        'sha256': '8dff5943f4e109c6d500763c803eaaeea1e8e4f67bdd3b270687acd2ec655873',
    }),
    ('faiss_index', {
        'path': CELL_7C0_DIR / 'cell_7c0_semantic_corpus_indexflatip_v1.faiss',
        'sha256': 'f676bca91e1c3dcdccb69eb76d626bf161cf81ae0eba05824e21141ddaf3718a',
    }),
    ('semantic_top20_pool', {
        'path': CELL_7C0_DIR / 'cell_7c0_common_semantic_top20_candidate_pool_v1.parquet',
        'sha256': 'be7f7cfc369952fc905cde5fb843e126b8e36d86f9bc625be853e9489d5327cb',
    }),
    ('execution_report', {
        'path': CELL_7C0_DIR / 'cell_7c0_embedding_and_semantic_retrieval_report_v1.json',
        'sha256': 'cf2402354699f0e69aa547efb67655145673d33de5fcb7c86281afc32c7282e2',
    }),
    ('qc', {
        'path': CELL_7C0_QC_DIR / 'cell_7c0_embedding_and_semantic_retrieval_qc_v1.json',
        'sha256': '732c1155fb49e95da8597b9d94d5c3f0d3da56b09dd9ec765758c5d9983e529a',
    }),
    ('manifest', {
        'path': CELL_7C0_CONFIG_DIR / 'cell_7c0_embedding_and_semantic_retrieval_manifest_v1.json',
        'sha256': '3b645237d8d5dfa04345f649b3dd2446e94abf10fb9e855e2c98d642cdd837e8',
    }),
])

# --------------------------------------------------------------------------------------------------
# Frozen Cell 7A3 score package
# --------------------------------------------------------------------------------------------------
CELL_7A3_SCORE_TABLE = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet'
)
CELL_7A3_MANIFEST = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7a3_t1_score_materialization_manifest_v1.json'
)
EXPECTED_CELL_7A3_SCORE_SHA256 = 'e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802'
EXPECTED_CELL_7A3_MANIFEST_SHA256 = '99bcff934f5e0c15d8357450fb3992eccfeb972ab61ddba8431a843c13b532dd'

# --------------------------------------------------------------------------------------------------
# Frozen Cell 7B4 ranking configuration
# --------------------------------------------------------------------------------------------------
CELL_7B4_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'
EMBEDDING_RETRIEVAL_CONFIG = CELL_7B4_DIR / 'cell_7b4_embedding_retrieval_configuration_v1.json'
QUALITY_CONFIG = CELL_7B4_DIR / 'cell_7b4_quality_reranking_configuration_v1.json'
CONDITION_ALIASES = CELL_7B4_DIR / 'cell_7b4_condition_alias_inventory_v1.csv'
CELL_7B4_MANIFEST = CELL_7B4_DIR / 'cell_7b4_configuration_freeze_manifest_v1.json'

EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256 = 'ab209e48b025652e5ddbb79891225334ffc51de62f157b430c21aef30865b807'
EXPECTED_QUALITY_CONFIG_SHA256 = '35c3871db6dd6bad9436e7202a67b064cc98beee06d19894e409d42f7ca006fb'
EXPECTED_CONDITION_ALIASES_SHA256 = '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9'
EXPECTED_CELL_7B4_MANIFEST_SHA256 = '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe'

# --------------------------------------------------------------------------------------------------
# Cell 7C2 outputs
# --------------------------------------------------------------------------------------------------
EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)

OUTPUTS = OrderedDict([
    ('ranking_audit',
     EXEC_DIR / 'cell_7c2_six_condition_quality_reranking_audit_v1.parquet'),
    ('score_blind_top5',
     EXEC_DIR / 'cell_7c2_score_blind_final_top5_context_selection_v1.parquet'),
    ('condition_summary',
     EXEC_DIR / 'cell_7c2_condition_reranking_summary_v1.csv'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c2_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c2_quality_reranking_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c2_quality_reranking_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c2_quality_reranking_and_top5_manifest_v1.json'),
])

for directory in (EXEC_DIR, QC_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing_outputs = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing_outputs:
    raise FileExistsError(
        'Cell 7C2 fail-closed overwrite protection is active. Existing frozen output(s):\n- '
        + '\n- '.join(existing_outputs)
    )

print(f'Execution directory : {EXEC_DIR}')
print(f'QC directory        : {QC_DIR}')
print(f'Config directory    : {CONFIG_DIR}')

Execution directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c2_quality_reranking_and_top5_v1
QC directory        : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c2_quality_reranking_and_top5_v1
Config directory    : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c2_quality_reranking_and_top5_v1


## 2. Cryptographic helpers and complete upstream authorization reverification

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    matches = re.findall(r'\b[a-fA-F0-9]{64}\b', text)
    if not matches:
        raise ValueError(f'No SHA-256 found in sidecar: {path}')
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\n'
            f'Expected: {expected_sha256}\n'
            f'Observed: {observed}\n'
            f'Path: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Missing or invalid SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'row_groups': int(pf.metadata.num_row_groups),
        'schema_names': list(pf.schema_arrow.names),
    }


verified_inputs: list[dict[str, Any]] = []

# Cell 7C1: exact V2 authorization outputs.
for key, spec in CELL_7C1.items():
    rec = verify_exact_artifact(f'cell_7c1_{key}', spec['path'], spec['sha256'])
    rec['source_cell'] = '7C1'
    verified_inputs.append(rec)

auth_7c1 = load_json(CELL_7C1['authorization']['path'])
qc_7c1 = load_json(CELL_7C1['qc']['path'])
manifest_7c1 = load_json(CELL_7C1['manifest']['path'])

if auth_7c1.get('authorization_decision') != EXPECTED_CELL_7C1_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C1 authorization decision does not exactly authorize the frozen Cell 7C2 scope.')
if auth_7c1.get('authorized_cell', {}).get('cell_id') != '7C2':
    raise AssertionError('Cell 7C1 authorization does not target Cell 7C2.')
if auth_7c1.get('authorized_cell', {}).get('authorized_once') is not True:
    raise AssertionError('Cell 7C1 authorization must be single-use/authorized_once=True.')
if int(qc_7c1.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C1 QC does not report zero failures.')
if manifest_7c1.get('terminal_decision') != EXPECTED_CELL_7C1_TERMINAL_DECISION:
    raise AssertionError('Cell 7C1 terminal PASS decision mismatch.')
if manifest_7c1.get('next_authorized_cell') != '7C2':
    raise AssertionError('Cell 7C1 manifest does not authorize Cell 7C2.')

# Complete Cell 7C0 package reverification.
for key, spec in CELL_7C0.items():
    rec = verify_exact_artifact(f'cell_7c0_{key}', spec['path'], spec['sha256'])
    rec['source_cell'] = '7C0'
    verified_inputs.append(rec)

manifest_7c0 = load_json(CELL_7C0['manifest']['path'])
if manifest_7c0.get('terminal_decision') != EXPECTED_CELL_7C0_DECISION:
    raise AssertionError('Cell 7C0 terminal PASS decision mismatch.')

# Cell 7A3 score table + manifest.
for label, path, expected_hash in [
    ('cell_7a3_score_table', CELL_7A3_SCORE_TABLE, EXPECTED_CELL_7A3_SCORE_SHA256),
    ('cell_7a3_manifest', CELL_7A3_MANIFEST, EXPECTED_CELL_7A3_MANIFEST_SHA256),
]:
    rec = verify_exact_artifact(label, path, expected_hash)
    rec['source_cell'] = '7A3'
    verified_inputs.append(rec)

manifest_7a3 = load_json(CELL_7A3_MANIFEST)
observed_7a3_decision = manifest_7a3.get('terminal_decision') or manifest_7a3.get('decision')
if observed_7a3_decision != EXPECTED_CELL_7A3_DECISION:
    raise AssertionError('Cell 7A3 terminal PASS decision mismatch.')

# Cell 7B4 exact configs + aliases + manifest.
for label, path, expected_hash in [
    ('cell_7b4_embedding_retrieval_config', EMBEDDING_RETRIEVAL_CONFIG, EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256),
    ('cell_7b4_quality_config', QUALITY_CONFIG, EXPECTED_QUALITY_CONFIG_SHA256),
    ('cell_7b4_condition_aliases', CONDITION_ALIASES, EXPECTED_CONDITION_ALIASES_SHA256),
    ('cell_7b4_manifest', CELL_7B4_MANIFEST, EXPECTED_CELL_7B4_MANIFEST_SHA256),
]:
    rec = verify_exact_artifact(label, path, expected_hash)
    rec['source_cell'] = '7B4'
    verified_inputs.append(rec)

manifest_7b4 = load_json(CELL_7B4_MANIFEST)
observed_7b4_decision = manifest_7b4.get('terminal_decision') or manifest_7b4.get('decision')
if observed_7b4_decision != EXPECTED_CELL_7B4_DECISION:
    raise AssertionError('Cell 7B4 terminal PASS decision mismatch.')

# Structural metadata checks before row-level score loading.
top20_meta = parquet_metadata(CELL_7C0['semantic_top20_pool']['path'])
score_meta = parquet_metadata(CELL_7A3_SCORE_TABLE)

EXPECTED_TOP20_SCHEMA = [
    'question_id',
    'semantic_rank',
    'semantic_score',
    'corpus_row_index',
    'packet_id',
    'rcv_accession',
]
if top20_meta['rows'] != EXPECTED_TOP20_ROWS:
    raise AssertionError(f'Frozen top-20 row count changed: {top20_meta["rows"]}')
if top20_meta['schema_names'] != EXPECTED_TOP20_SCHEMA:
    raise AssertionError(
        'Frozen Cell 7C0 top-20 schema changed.\n'
        f'Observed: {top20_meta["schema_names"]}'
    )

EXPECTED_SCORE_SCHEMA = [
    't1_row_order',
    'rcv_accession',
    'target_gene',
    'classification_axis',
    'review_stars_instability_risk',
    'conflict_instability_risk',
    'recency_instability_risk',
    'recency_missing_instability_component',
    'submitter_instability_risk',
    'entropy_instability_risk',
    'combined_metadata_instability_risk',
    'full_ges_p_stable_t1',
    'full_ges_instability_risk_t1',
    'no_star_ges_p_stable_t1',
    'no_star_ges_instability_risk_t1',
    'comparator_policy_version',
    'comparator_policy_sha256',
    'cell_7a2_manifest_sha256',
    'stage7a3_version',
    'model_fitted_or_refitted',
    'threshold_or_weight_optimized',
    'score_rank_constructed',
    'hard_exclusion_applied',
    'outcome_labels_loaded',
    'temporal_performance_evaluated',
    'rag_corpus_constructed',
    'embeddings_constructed',
    'llm_called',
]
if score_meta['rows'] != 100_920 or score_meta['columns'] != 28:
    raise AssertionError(
        f'Cell 7A3 score-table metadata changed: {score_meta["rows"]:,} × {score_meta["columns"]}'
    )
if score_meta['schema_names'] != EXPECTED_SCORE_SCHEMA:
    raise AssertionError(
        'Cell 7A3 score-table schema/order changed.\n'
        f'Observed: {score_meta["schema_names"]}'
    )

print('Cell 7C1 authorization package : VERIFIED')
print('Cell 7C0 complete package      : 10/10 VERIFIED')
print(f'Cell 7C0 top-20 metadata       : {top20_meta["rows"]:,} rows × {top20_meta["columns"]} columns')
print(f'Cell 7A3 score metadata        : {score_meta["rows"]:,} rows × {score_meta["columns"]} columns')
print('Cell 7B4 ranking configuration : VERIFIED')
print('Row-level Cell 7A3 scores      : NOT YET LOADED')
print('Answer-key outcomes            : NOT LOADED')

Cell 7C1 authorization package : VERIFIED
Cell 7C0 complete package      : 10/10 VERIFIED
Cell 7C0 top-20 metadata       : 1,600 rows × 6 columns
Cell 7A3 score metadata        : 100,920 rows × 28 columns
Cell 7B4 ranking configuration : VERIFIED
Row-level Cell 7A3 scores      : NOT YET LOADED
Answer-key outcomes            : NOT LOADED


## 3. Validate the exact frozen Cell 7B4 ranking implementation

In [ ]:
embedding_config = load_json(EMBEDDING_RETRIEVAL_CONFIG)
quality_config = load_json(QUALITY_CONFIG)

with CONDITION_ALIASES.open('r', encoding='utf-8', newline='') as handle:
    alias_rows = list(csv.DictReader(handle))

if len(alias_rows) != 6:
    raise AssertionError(f'Expected six blinded aliases; observed {len(alias_rows)}.')

alias_map = {row['condition_id']: row['blinded_alias'] for row in alias_rows}
expected_alias_map = OrderedDict([
    ('A', 'ARM-MICA'),
    ('B', 'ARM-ORBIT'),
    ('C', 'ARM-KITE'),
    ('D', 'ARM-PULSE'),
    ('E', 'ARM-LARCH'),
    ('F', 'ARM-NOVA'),
])
if alias_map != dict(expected_alias_map):
    raise AssertionError(f'Frozen blinded aliases changed: {alias_map}')

candidate_pool = embedding_config.get('candidate_pool', {})
rerank_pool = quality_config.get('candidate_pool_invariance', {})
quality_rank_cfg = quality_config.get('quality_rank', {})
rrf_cfg = quality_config.get('weighted_reciprocal_rank_fusion', {})
random_cfg = quality_config.get('random_quality_control', {})
condition_a_cfg = quality_config.get('condition_a_semantic_only', {})
conditions_bf_cfg = quality_config.get('conditions_b_to_f', {})
condition_inventory = quality_config.get('condition_inventory', [])

EXPECTED_RANDOM_REFERENCE = '''def deterministic_random_quality(question_id: str, rcv_accession: str) -> float:
    payload = f"20260722|{question_id}|{rcv_accession}".encode("utf-8")
    integer = int(hashlib.sha256(payload).hexdigest()[:16], 16)
    return integer / float((1 << 64) - 1)'''

EXPECTED_RRF_REFERENCE = '''def weighted_rrf(semantic_rank: int, quality_rank: int) -> float:
    return 0.75 / (60 + semantic_rank) + 0.25 / (60 + quality_rank)'''

configuration_checks = OrderedDict([
    ('semantic_candidate_pool_k_20', candidate_pool.get('semantic_candidate_pool_k') == 20),
    ('final_context_k_5', candidate_pool.get('final_context_k') == 5),
    ('same_top20_pool_all_conditions', candidate_pool.get('same_top20_pool_for_all_conditions') is True),
    ('candidate_pool_reused_without_mutation', candidate_pool.get('candidate_pool_reused_without_mutation') is True),
    ('hard_exclusion_false_embedding_config', candidate_pool.get('hard_evidence_exclusion') is False),
    ('score_threshold_filtering_false', candidate_pool.get('score_threshold_filtering') is False),
    ('rerank_input_pool_exact', rerank_pool.get('input_pool') == 'same frozen semantic top-20 for all six conditions'),
    ('rerank_input_order_exact', rerank_pool.get('input_order') == 'semantic rank ascending'),
    ('rerank_final_context_k_5', rerank_pool.get('final_context_k') == 5),
    ('rerank_hard_exclusion_false', rerank_pool.get('hard_exclusion') is False),
    ('quality_score_not_exposed_to_llm', rerank_pool.get('quality_score_exposed_to_llm') is False),
    ('quality_rank_scope_exact', quality_rank_cfg.get('ranking_scope') == "within each question's fixed top-20 candidate pool"),
    ('quality_rank_direction_exact', quality_rank_cfg.get('direction') == 'higher quality signal receives better rank'),
    ('quality_rank_origin_1', quality_rank_cfg.get('rank_origin') == 1),
    ('quality_rank_ties_exact',
     quality_rank_cfg.get('tie_breaker_order') == ['rcv_accession ascending', 'packet_id ascending']),
    ('rrf_formula_exact',
     rrf_cfg.get('formula') == '0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)'),
    ('rrf_semantic_weight_075', rrf_cfg.get('semantic_weight') == 0.75),
    ('rrf_quality_weight_025', rrf_cfg.get('quality_weight') == 0.25),
    ('rrf_constant_60', rrf_cfg.get('rrf_constant') == 60),
    ('rrf_descending_sort', rrf_cfg.get('descending_sort') is True),
    ('rrf_ties_exact',
     rrf_cfg.get('final_tie_breaker_order') == [
         'semantic_rank ascending',
         'quality_rank ascending',
         'rcv_accession ascending',
         'packet_id ascending',
     ]),
    ('condition_a_quality_rank_false', condition_a_cfg.get('quality_rank_constructed') is False),
    ('condition_a_rrf_false', condition_a_cfg.get('rrf_applied') is False),
    ('condition_a_final_exact', condition_a_cfg.get('final_context') == 'semantic ranks 1 through 5'),
    ('conditions_bf_rrf_true', conditions_bf_cfg.get('rrf_applied') is True),
    ('conditions_bf_final_exact', conditions_bf_cfg.get('final_context') == 'top five by frozen weighted RRF'),
    ('random_seed_exact', random_cfg.get('seed') == FROZEN_RANDOM_SEED),
    ('random_hash_exact', random_cfg.get('hash_algorithm') == 'SHA-256'),
    ('random_payload_exact', random_cfg.get('payload') == '20260722|{question_id}|{rcv_accession}'),
    ('random_integer_source_exact',
     random_cfg.get('integer_source') == 'first 16 hexadecimal characters interpreted as unsigned 64-bit integer'),
    ('random_scale_exact', random_cfg.get('scale') == 'integer / (2^64 - 1)'),
    ('random_reference_exact', random_cfg.get('reference_implementation') == EXPECTED_RANDOM_REFERENCE),
    ('rrf_reference_exact', rrf_cfg.get('reference_implementation') == EXPECTED_RRF_REFERENCE),
    ('six_conditions_frozen', len(condition_inventory) == 6),
    ('primary_comparison_d_vs_a', quality_config.get('primary_comparison') == 'D_vs_A'),
])

failed_configuration_checks = [
    name for name, passed in configuration_checks.items() if not bool(passed)
]
if failed_configuration_checks:
    raise RuntimeError(
        'Frozen Cell 7B4 ranking configuration verification failed:\n- '
        + '\n- '.join(failed_configuration_checks)
    )

inventory_by_id = {row['condition_id']: row for row in condition_inventory}
expected_quality_signals = {
    'A': 'none',
    'B': '1 - mean(review_stars_instability_risk, conflict_instability_risk)',
    'C': '1 - combined_metadata_instability_risk',
    'D': 'full_ges_p_stable_t1',
    'E': 'no_star_ges_p_stable_t1',
    'F': 'deterministic SHA-256 pseudo-random quality score',
}
for condition_id, expected_signal in expected_quality_signals.items():
    if inventory_by_id.get(condition_id, {}).get('quality_signal') != expected_signal:
        raise AssertionError(
            f'Condition {condition_id} quality signal changed. '
            f'Observed: {inventory_by_id.get(condition_id, {}).get("quality_signal")}'
        )

def deterministic_random_quality(question_id: str, rcv_accession: str) -> float:
    payload = f'20260722|{question_id}|{rcv_accession}'.encode('utf-8')
    integer = int(hashlib.sha256(payload).hexdigest()[:16], 16)
    return integer / float((1 << 64) - 1)


def weighted_rrf(semantic_rank: int | np.ndarray, quality_rank: int | np.ndarray):
    return 0.75 / (60 + semantic_rank) + 0.25 / (60 + quality_rank)


print('Frozen ranking design                    : VERIFIED')
print('Semantic candidate pool                  : 20')
print('Final context                            : 5')
print('RRF                                      : 0.75 semantic + 0.25 quality; constant 60')
print('Random-quality seed                      : 20260722')
print('Hard evidence exclusion                  : FALSE')
print('Score exposure to LLM                    : FALSE')

Frozen ranking design                    : VERIFIED
Semantic candidate pool                  : 20
Final context                            : 5
RRF                                      : 0.75 semantic + 0.25 quality; constant 60
Random-quality seed                      : 20260722
Hard evidence exclusion                  : FALSE
Score exposure to LLM                    : FALSE


## 4. Load only the authorized row-level inputs and verify the score-to-candidate join

In [ ]:
# Snapshot immutable source hashes before authorized row-level loading.
IMMUTABLE_SOURCE_PATHS = OrderedDict([
    ('cell_7c1_authorization', CELL_7C1['authorization']['path']),
    ('cell_7c1_manifest', CELL_7C1['manifest']['path']),
    ('cell_7c0_top20', CELL_7C0['semantic_top20_pool']['path']),
    ('cell_7c0_manifest', CELL_7C0['manifest']['path']),
    ('cell_7a3_scores', CELL_7A3_SCORE_TABLE),
    ('cell_7a3_manifest', CELL_7A3_MANIFEST),
    ('cell_7b4_embedding_config', EMBEDDING_RETRIEVAL_CONFIG),
    ('cell_7b4_quality_config', QUALITY_CONFIG),
    ('cell_7b4_aliases', CONDITION_ALIASES),
    ('cell_7b4_manifest', CELL_7B4_MANIFEST),
])
immutable_hashes_before = OrderedDict(
    (key, sha256_file(path)) for key, path in IMMUTABLE_SOURCE_PATHS.items()
)

top20 = pd.read_parquet(CELL_7C0['semantic_top20_pool']['path'])
scores = pd.read_parquet(CELL_7A3_SCORE_TABLE)

# Exact Cell 7C0 candidate-pool structure.
if list(top20.columns) != EXPECTED_TOP20_SCHEMA:
    raise AssertionError('Loaded top-20 column order changed.')
if len(top20) != EXPECTED_TOP20_ROWS:
    raise AssertionError(f'Expected {EXPECTED_TOP20_ROWS:,} top-20 rows; observed {len(top20):,}.')
if top20['question_id'].nunique(dropna=False) != EXPECTED_QUESTIONS:
    raise AssertionError('Frozen top-20 pool does not contain exactly 80 questions.')
if top20[['question_id', 'packet_id']].duplicated().any():
    raise AssertionError('Duplicate question/packet pair found in frozen top-20 pool.')
if top20[['question_id', 'rcv_accession']].duplicated().any():
    raise AssertionError('Duplicate question/RCV pair found in frozen top-20 pool.')
if top20['semantic_score'].isna().any() or not np.isfinite(top20['semantic_score'].to_numpy(dtype=float)).all():
    raise AssertionError('Non-finite semantic score found in frozen top-20 pool.')

rank_counts = top20.groupby('question_id', sort=False).size()
if not rank_counts.eq(EXPECTED_TOP20_PER_QUESTION).all():
    raise AssertionError('Every question must have exactly 20 frozen semantic candidates.')

rank_sets = top20.groupby('question_id')['semantic_rank'].apply(lambda s: tuple(sorted(int(v) for v in s.tolist())))
if not rank_sets.map(lambda values: values == tuple(range(1, 21))).all():
    raise AssertionError('Every question must contain semantic ranks exactly 1 through 20.')

# Exact Cell 7A3 score structure and governance flags.
if list(scores.columns) != EXPECTED_SCORE_SCHEMA:
    raise AssertionError('Loaded Cell 7A3 score schema/order changed.')
if len(scores) != 100_920:
    raise AssertionError(f'Expected 100,920 Cell 7A3 rows; observed {len(scores):,}.')
if scores['rcv_accession'].isna().any() or scores['rcv_accession'].duplicated().any():
    raise AssertionError('Cell 7A3 RCV accession key is missing or duplicated.')

QUALITY_SOURCE_COLUMNS = [
    'review_stars_instability_risk',
    'conflict_instability_risk',
    'combined_metadata_instability_risk',
    'full_ges_p_stable_t1',
    'no_star_ges_p_stable_t1',
]
for column in QUALITY_SOURCE_COLUMNS:
    values = scores[column].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError(f'Non-finite values in authorized score column: {column}')
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f'Out-of-range values in authorized score column: {column}')

AUDIT_FALSE_COLUMNS = [
    'model_fitted_or_refitted',
    'threshold_or_weight_optimized',
    'score_rank_constructed',
    'hard_exclusion_applied',
    'outcome_labels_loaded',
    'temporal_performance_evaluated',
    'rag_corpus_constructed',
    'embeddings_constructed',
    'llm_called',
]
for column in AUDIT_FALSE_COLUMNS:
    if not scores[column].eq(False).all():
        raise AssertionError(f'Cell 7A3 governance flag must remain FALSE on every row: {column}')

if not np.allclose(
    scores['full_ges_p_stable_t1'].to_numpy(dtype=float)
    + scores['full_ges_instability_risk_t1'].to_numpy(dtype=float),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise AssertionError('Full-GES P(stable)/instability complement check failed.')

if not np.allclose(
    scores['no_star_ges_p_stable_t1'].to_numpy(dtype=float)
    + scores['no_star_ges_instability_risk_t1'].to_numpy(dtype=float),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise AssertionError('No-star-GES P(stable)/instability complement check failed.')

score_subset = scores[
    [
        'rcv_accession',
        'review_stars_instability_risk',
        'conflict_instability_risk',
        'combined_metadata_instability_risk',
        'full_ges_p_stable_t1',
        'no_star_ges_p_stable_t1',
    ]
].copy()

joined = top20.merge(
    score_subset,
    on='rcv_accession',
    how='left',
    validate='many_to_one',
    indicator=True,
    sort=False,
)

if len(joined) != EXPECTED_TOP20_ROWS:
    raise AssertionError('Score join changed the frozen candidate-row count.')
if not joined['_merge'].eq('both').all():
    missing = int(joined['_merge'].ne('both').sum())
    raise AssertionError(f'{missing} frozen candidate rows failed to join to Cell 7A3 scores.')
joined = joined.drop(columns=['_merge'])

if joined[QUALITY_SOURCE_COLUMNS].isna().any().any():
    raise AssertionError('At least one frozen top-20 candidate is missing an authorized quality signal after join.')

# Exact frozen candidate order as a reusable comparison frame.
source_candidate_identity = (
    top20[
        ['question_id', 'semantic_rank', 'semantic_score', 'corpus_row_index', 'packet_id', 'rcv_accession']
    ]
    .sort_values(['question_id', 'semantic_rank'], kind='mergesort')
    .reset_index(drop=True)
)

print(f'Cell 7A3 score rows loaded                : {len(scores):,}')
print(f'Frozen semantic candidates loaded         : {len(top20):,}')
print(f'Authorized candidate-score joins           : {len(joined):,}/{len(top20):,}')
print('Candidate rows dropped or added            : 0')
print('Model fitting/refitting                    : NO')
print('Threshold/weight optimization              : NO')
print('Answer keys/outcomes loaded                : NO')

Cell 7A3 score rows loaded                : 100,920
Frozen semantic candidates loaded         : 1,600
Authorized candidate-score joins           : 1,600/1,600
Candidate rows dropped or added            : 0
Model fitting/refitting                    : NO
Threshold/weight optimization              : NO
Answer keys/outcomes loaded                : NO


## 5. Construct the six frozen conditions, deterministic quality ranks, fixed RRF, and final top-5

In [ ]:
CONDITION_ORDER = ['A', 'B', 'C', 'D', 'E', 'F']

def build_condition_frame(base: pd.DataFrame, condition_id: str) -> pd.DataFrame:
    frame = base[
        [
            'question_id',
            'semantic_rank',
            'semantic_score',
            'corpus_row_index',
            'packet_id',
            'rcv_accession',
            'review_stars_instability_risk',
            'conflict_instability_risk',
            'combined_metadata_instability_risk',
            'full_ges_p_stable_t1',
            'no_star_ges_p_stable_t1',
        ]
    ].copy()

    inv = inventory_by_id[condition_id]
    frame.insert(0, 'condition_id', condition_id)
    frame.insert(1, 'blinded_alias', alias_map[condition_id])
    frame.insert(2, 'condition_name', inv['condition_name'])
    frame.insert(3, 'condition_role', inv['role'])
    frame.insert(4, 'quality_signal_definition', inv['quality_signal'])
    frame.insert(5, 'final_order_policy', inv['final_order_policy'])

    if condition_id == 'A':
        frame['quality_signal'] = np.nan
        frame['quality_rank'] = pd.Series(pd.NA, index=frame.index, dtype='Int64')
        frame['rrf_score'] = np.nan
        frame['final_context_rank'] = frame['semantic_rank'].astype(np.int64)
        frame['selected_top5'] = frame['final_context_rank'].le(5)
        return frame

    if condition_id == 'B':
        frame['quality_signal'] = 1.0 - (
            frame['review_stars_instability_risk'].astype(float)
            + frame['conflict_instability_risk'].astype(float)
        ) / 2.0
    elif condition_id == 'C':
        frame['quality_signal'] = 1.0 - frame['combined_metadata_instability_risk'].astype(float)
    elif condition_id == 'D':
        frame['quality_signal'] = frame['full_ges_p_stable_t1'].astype(float)
    elif condition_id == 'E':
        frame['quality_signal'] = frame['no_star_ges_p_stable_t1'].astype(float)
    elif condition_id == 'F':
        frame['quality_signal'] = [
            deterministic_random_quality(str(q), str(r))
            for q, r in zip(frame['question_id'], frame['rcv_accession'])
        ]
        # Independent second computation to fail closed on accidental nondeterminism.
        random_repeat = np.array([
            deterministic_random_quality(str(q), str(r))
            for q, r in zip(frame['question_id'], frame['rcv_accession'])
        ], dtype=float)
        if not np.array_equal(frame['quality_signal'].to_numpy(dtype=float), random_repeat):
            raise AssertionError('Deterministic random-quality reproduction failed.')
    else:
        raise ValueError(f'Unknown frozen condition: {condition_id}')

    q_values = frame['quality_signal'].to_numpy(dtype=float)
    if not np.isfinite(q_values).all():
        raise AssertionError(f'Condition {condition_id} produced non-finite quality signals.')
    if ((q_values < 0.0) | (q_values > 1.0)).any():
        raise AssertionError(f'Condition {condition_id} produced quality signals outside [0,1].')

    # Frozen quality rank: higher quality first; deterministic RCV then packet tie-break.
    frame = frame.sort_values(
        ['question_id', 'quality_signal', 'rcv_accession', 'packet_id'],
        ascending=[True, False, True, True],
        kind='mergesort',
    ).copy()
    frame['quality_rank'] = (
        frame.groupby('question_id', sort=False).cumcount().add(1).astype('Int64')
    )

    # Exact frozen weighted RRF.
    frame['rrf_score'] = weighted_rrf(
        frame['semantic_rank'].to_numpy(dtype=float),
        frame['quality_rank'].astype(np.int64).to_numpy(dtype=float),
    )

    # Frozen final RRF ordering and tie-break.
    frame = frame.sort_values(
        [
            'question_id',
            'rrf_score',
            'semantic_rank',
            'quality_rank',
            'rcv_accession',
            'packet_id',
        ],
        ascending=[True, False, True, True, True, True],
        kind='mergesort',
    ).copy()

    frame['final_context_rank'] = (
        frame.groupby('question_id', sort=False).cumcount().add(1).astype(np.int64)
    )
    frame['selected_top5'] = frame['final_context_rank'].le(5)
    return frame


condition_frames = [build_condition_frame(joined, condition_id) for condition_id in CONDITION_ORDER]
ranking_audit = pd.concat(condition_frames, ignore_index=True)

# Keep deterministic artifact ordering.
condition_order_map = {condition_id: i for i, condition_id in enumerate(CONDITION_ORDER)}
ranking_audit['_condition_order'] = ranking_audit['condition_id'].map(condition_order_map).astype(np.int64)
ranking_audit = ranking_audit.sort_values(
    ['question_id', '_condition_order', 'final_context_rank'],
    kind='mergesort',
).drop(columns=['_condition_order']).reset_index(drop=True)

# Remove raw source-component columns from the persisted audit once quality_signal is frozen.
# This keeps the audit sufficient for exact reranking verification while minimizing score-bearing payload.
ranking_audit = ranking_audit[
    [
        'question_id',
        'condition_id',
        'blinded_alias',
        'condition_name',
        'condition_role',
        'quality_signal_definition',
        'final_order_policy',
        'semantic_rank',
        'semantic_score',
        'corpus_row_index',
        'packet_id',
        'rcv_accession',
        'quality_signal',
        'quality_rank',
        'rrf_score',
        'final_context_rank',
        'selected_top5',
    ]
].copy()

# Score-blind selection for the later prompt stage.
score_blind_top5 = (
    ranking_audit.loc[
        ranking_audit['selected_top5'],
        [
            'question_id',
            'blinded_alias',
            'final_context_rank',
            'corpus_row_index',
            'packet_id',
            'rcv_accession',
        ],
    ]
    .rename(columns={'final_context_rank': 'context_position'})
    .copy()
)
score_blind_top5['_alias_order'] = score_blind_top5['blinded_alias'].map(
    {alias: i for i, alias in enumerate(expected_alias_map.values())}
).astype(np.int64)
score_blind_top5 = score_blind_top5.sort_values(
    ['question_id', '_alias_order', 'context_position'],
    kind='mergesort',
).drop(columns=['_alias_order']).reset_index(drop=True)

# Minimal non-performance summary only.
condition_summary = pd.DataFrame([
    {
        'condition_id': condition_id,
        'blinded_alias': alias_map[condition_id],
        'condition_name': inventory_by_id[condition_id]['condition_name'],
        'condition_role': inventory_by_id[condition_id]['role'],
        'quality_signal_definition': inventory_by_id[condition_id]['quality_signal'],
        'final_order_policy': inventory_by_id[condition_id]['final_order_policy'],
        'questions': EXPECTED_QUESTIONS,
        'candidate_rows': EXPECTED_QUESTIONS * EXPECTED_TOP20_PER_QUESTION,
        'final_top5_rows': EXPECTED_QUESTIONS * EXPECTED_TOP5_PER_QUESTION_CONDITION,
        'hard_exclusion_applied': False,
        'prompt_materialized': False,
        'llm_called': False,
    }
    for condition_id in CONDITION_ORDER
])

print(f'Condition-specific audit rows             : {len(ranking_audit):,}')
print(f'Score-blind final top-5 rows              : {len(score_blind_top5):,}')
print(f'Conditions                                : {ranking_audit["condition_id"].nunique()}')
print('Quality rankings constructed              : B/C/D/E/F = YES; A = NO')
print('Fixed RRF applied                         : B/C/D/E/F = YES; A = NO')
print('Hard evidence exclusion                   : NO')
print('Prompts materialized                      : NO')
print('LLM called                                : NO')

Condition-specific audit rows             : 9,600
Score-blind final top-5 rows              : 2,400
Conditions                                : 6
Quality rankings constructed              : B/C/D/E/F = YES; A = NO
Fixed RRF applied                         : B/C/D/E/F = YES; A = NO
Hard evidence exclusion                   : NO
Prompts materialized                      : NO
LLM called                                : NO


## 6. Fail-closed scientific and leakage QC before writing any frozen Cell 7C2 artifact

In [ ]:
prewrite_checks = OrderedDict()

# Global row and group accounting.
prewrite_checks['ranking_audit_rows_9600'] = len(ranking_audit) == EXPECTED_AUDIT_ROWS
prewrite_checks['ranking_audit_six_conditions'] = ranking_audit['condition_id'].nunique() == EXPECTED_CONDITIONS
prewrite_checks['ranking_audit_80_questions'] = ranking_audit['question_id'].nunique() == EXPECTED_QUESTIONS
prewrite_checks['exact_20_rows_per_question_condition'] = (
    ranking_audit.groupby(['question_id', 'condition_id']).size().eq(20).all()
)
prewrite_checks['exact_final_ranks_1_to_20'] = (
    ranking_audit.groupby(['question_id', 'condition_id'])['final_context_rank']
    .apply(lambda s: tuple(sorted(int(v) for v in s.tolist())) == tuple(range(1, 21)))
    .all()
)
prewrite_checks['score_blind_top5_rows_2400'] = len(score_blind_top5) == EXPECTED_FINAL_TOP5_ROWS
prewrite_checks['exact_5_top5_rows_per_question_alias'] = (
    score_blind_top5.groupby(['question_id', 'blinded_alias']).size().eq(5).all()
)
prewrite_checks['score_blind_context_positions_1_to_5'] = (
    score_blind_top5.groupby(['question_id', 'blinded_alias'])['context_position']
    .apply(lambda s: tuple(sorted(int(v) for v in s.tolist())) == (1, 2, 3, 4, 5))
    .all()
)

# Candidate-pool invariance: every condition contains the exact same 20 source candidates.
source_sets = (
    source_candidate_identity.groupby('question_id')
    .apply(lambda g: frozenset(zip(g['packet_id'].astype(str), g['rcv_accession'].astype(str))))
)
candidate_pool_invariant = True
for condition_id in CONDITION_ORDER:
    cond = ranking_audit.loc[ranking_audit['condition_id'].eq(condition_id)]
    observed_sets = (
        cond.groupby('question_id')
        .apply(lambda g: frozenset(zip(g['packet_id'].astype(str), g['rcv_accession'].astype(str))))
    )
    if not observed_sets.index.equals(source_sets.index):
        candidate_pool_invariant = False
        break
    if not all(observed_sets.loc[q] == source_sets.loc[q] for q in source_sets.index):
        candidate_pool_invariant = False
        break
prewrite_checks['all_conditions_exact_frozen_top20_membership'] = candidate_pool_invariant

# Condition A must be exact semantic ranks 1-5 and must not have quality/RRF values.
cond_a = ranking_audit.loc[ranking_audit['condition_id'].eq('A')].copy()
prewrite_checks['condition_a_quality_signal_null'] = cond_a['quality_signal'].isna().all()
prewrite_checks['condition_a_quality_rank_null'] = cond_a['quality_rank'].isna().all()
prewrite_checks['condition_a_rrf_null'] = cond_a['rrf_score'].isna().all()
prewrite_checks['condition_a_final_rank_equals_semantic_rank'] = (
    cond_a['final_context_rank'].astype(int).to_numpy()
    == cond_a['semantic_rank'].astype(int).to_numpy()
).all()
prewrite_checks['condition_a_top5_exact_semantic_1_to_5'] = (
    cond_a.loc[cond_a['selected_top5'], 'semantic_rank'].between(1, 5).all()
    and len(cond_a.loc[cond_a['selected_top5']]) == 400
)

# B-F quality ranks, RRF formula, and deterministic ordering.
for condition_id in ['B', 'C', 'D', 'E', 'F']:
    cond = ranking_audit.loc[ranking_audit['condition_id'].eq(condition_id)].copy()
    prewrite_checks[f'condition_{condition_id}_quality_nonmissing_finite'] = (
        cond['quality_signal'].notna().all()
        and np.isfinite(cond['quality_signal'].to_numpy(dtype=float)).all()
    )
    prewrite_checks[f'condition_{condition_id}_quality_rank_1_to_20'] = (
        cond.groupby('question_id')['quality_rank']
        .apply(lambda s: tuple(sorted(int(v) for v in s.tolist())) == tuple(range(1, 21)))
        .all()
    )
    exact_rrf = weighted_rrf(
        cond['semantic_rank'].to_numpy(dtype=float),
        cond['quality_rank'].astype(int).to_numpy(dtype=float),
    )
    prewrite_checks[f'condition_{condition_id}_rrf_exact'] = np.array_equal(
        cond['rrf_score'].to_numpy(dtype=float),
        np.asarray(exact_rrf, dtype=float),
    )
    prewrite_checks[f'condition_{condition_id}_top5_400'] = int(cond['selected_top5'].sum()) == 400

# Independently reconstruct each B-F quality signal from the joined frozen sources.
source_key = joined.set_index(['question_id', 'packet_id'], verify_integrity=True)

def aligned_source_values(cond: pd.DataFrame, column: str) -> np.ndarray:
    idx = pd.MultiIndex.from_arrays(
        [cond['question_id'].to_numpy(), cond['packet_id'].to_numpy()],
        names=['question_id', 'packet_id'],
    )
    return source_key.loc[idx, column].to_numpy(dtype=float)

cond_b = ranking_audit.loc[ranking_audit['condition_id'].eq('B')]
b_expected = 1.0 - (
    aligned_source_values(cond_b, 'review_stars_instability_risk')
    + aligned_source_values(cond_b, 'conflict_instability_risk')
) / 2.0
prewrite_checks['condition_b_quality_formula_exact'] = np.array_equal(
    cond_b['quality_signal'].to_numpy(dtype=float), b_expected
)

cond_c = ranking_audit.loc[ranking_audit['condition_id'].eq('C')]
c_expected = 1.0 - aligned_source_values(cond_c, 'combined_metadata_instability_risk')
prewrite_checks['condition_c_quality_formula_exact'] = np.array_equal(
    cond_c['quality_signal'].to_numpy(dtype=float), c_expected
)

cond_d = ranking_audit.loc[ranking_audit['condition_id'].eq('D')]
d_expected = aligned_source_values(cond_d, 'full_ges_p_stable_t1')
prewrite_checks['condition_d_quality_formula_exact'] = np.array_equal(
    cond_d['quality_signal'].to_numpy(dtype=float), d_expected
)

cond_e = ranking_audit.loc[ranking_audit['condition_id'].eq('E')]
e_expected = aligned_source_values(cond_e, 'no_star_ges_p_stable_t1')
prewrite_checks['condition_e_quality_formula_exact'] = np.array_equal(
    cond_e['quality_signal'].to_numpy(dtype=float), e_expected
)

cond_f = ranking_audit.loc[ranking_audit['condition_id'].eq('F')]
f_expected = np.array([
    deterministic_random_quality(str(q), str(r))
    for q, r in zip(cond_f['question_id'], cond_f['rcv_accession'])
], dtype=float)
prewrite_checks['condition_f_random_quality_exact'] = np.array_equal(
    cond_f['quality_signal'].to_numpy(dtype=float), f_expected
)

# Score-blind artifact must contain no score/rank leakage except context position.
PROHIBITED_SCORE_BLIND_TOKENS = (
    'ges',
    'stable',
    'instability',
    'quality',
    'semantic_rank',
    'semantic_score',
    'rrf',
    'review',
    'conflict',
    'metadata_score',
    'condition_id',
    'condition_name',
    'condition_role',
)
score_blind_columns_lower = [str(c).lower() for c in score_blind_top5.columns]
leaking_columns = [
    column
    for column in score_blind_columns_lower
    if any(token in column for token in PROHIBITED_SCORE_BLIND_TOKENS)
]
prewrite_checks['score_blind_top5_has_no_score_or_condition_identity_columns'] = len(leaking_columns) == 0
prewrite_checks['score_blind_top5_exact_columns'] = list(score_blind_top5.columns) == [
    'question_id',
    'blinded_alias',
    'context_position',
    'corpus_row_index',
    'packet_id',
    'rcv_accession',
]
prewrite_checks['score_blind_aliases_only'] = set(score_blind_top5['blinded_alias']) == set(expected_alias_map.values())

# No answer keys, prompts, generation, or metrics are touched by this notebook.
prewrite_checks['no_hard_exclusion'] = int(ranking_audit.shape[0]) == EXPECTED_AUDIT_ROWS
prewrite_checks['no_prompt_columns_materialized'] = not any('prompt' in c.lower() for c in ranking_audit.columns)
prewrite_checks['no_llm_output_columns_materialized'] = not any('llm' in c.lower() for c in ranking_audit.columns)
prewrite_checks['no_answer_key_columns_materialized'] = not any(
    'answer_key' in c.lower() or 'gold' in c.lower() for c in ranking_audit.columns
)
prewrite_checks['no_metric_columns_materialized'] = not any(
    token in c.lower()
    for c in ranking_audit.columns
    for token in ('accuracy', 'precision', 'recall', 'f1', 'auroc', 'auprc', 'brier', 'ndcg', 'mrr')
)

failed_prewrite_checks = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed_prewrite_checks:
    raise RuntimeError(
        'Cell 7C2 failed before writing any frozen output. Failed checks:\n- '
        + '\n- '.join(failed_prewrite_checks)
    )

print(f'Prewrite QC checks                         : {len(prewrite_checks)}/{len(prewrite_checks)} PASS')
print('Exact common top-20 membership             : PRESERVED IN ALL 6 CONDITIONS')
print('Condition A semantic-only top-5            : VERIFIED')
print('Conditions B-F quality ranking + RRF        : VERIFIED')
print('Score-blind top-5 leakage boundary          : VERIFIED')
print('Scientific outputs written so far           : NO')

Prewrite QC checks                         : 47/47 PASS
Exact common top-20 membership             : PRESERVED IN ALL 6 CONDITIONS
Condition A semantic-only top-5            : VERIFIED
Conditions B-F quality ranking + RRF        : VERIFIED
Score-blind top-5 leakage boundary          : VERIFIED
Scientific outputs written so far           : NO


/tmp/ipykernel_4242/2075460328.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: frozenset(zip(g['packet_id'].astype(str), g['rcv_accession'].astype(str))))
/tmp/ipykernel_4242/2075460328.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: frozenset(zip(g['packet_id'].astype(str), g['rcv_accession'].astype(str))))
/tmp/ipykernel_4242/2075460328.py:35: DeprecationWarning

## 7. Freeze Cell 7C2 artifacts, SHA-256 sidecars, QC, execution report, and manifest

In [ ]:
def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(payload, sort_keys=True, indent=2, ensure_ascii=False) + '\n'
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator='\n')
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(
        path,
        index=False,
        engine='pyarrow',
        compression='zstd',
    )
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(f'{digest}  {path.name}\n', encoding='utf-8')


input_inventory = pd.DataFrame([
    {
        'input_id': record['input_id'],
        'source_cell': record['source_cell'],
        'path': record['path'],
        'sha256': record['sha256'],
        'bytes': record['bytes'],
        'sidecar_path': record['sidecar_path'],
        'sidecar_valid': record['sidecar_valid'],
        'row_level_content_loaded_in_cell_7c2': (
            record['input_id'] in {
                'cell_7c0_semantic_top20_pool',
                'cell_7a3_score_table',
                'cell_7b4_quality_config',
                'cell_7b4_condition_aliases',
            }
        ),
    }
    for record in verified_inputs
])

terminal_decision = (
    'PASS_STAGE7C2_FROZEN_CELL7A3_SCORES_JOINED_TO_EXACT_CELL7C0_TOP20_'
    'SIX_CONDITION_QUALITY_RANKS_FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_'
    'AND_2400_SCORE_BLIND_FINAL_TOP5_SELECTIONS_MATERIALIZED_CHECKSUM_PROTECTED_'
    'NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS_'
    'NEXT_EXECUTION_NOT_AUTHORIZED'
)

with tempfile.TemporaryDirectory(prefix='cell_7c2_staging_') as tmpdir_text:
    tmpdir = Path(tmpdir_text)
    staged = {
        key: tmpdir / path.name
        for key, path in OUTPUTS.items()
    }

    # Scientific artifacts first.
    stable_write_parquet(staged['ranking_audit'], ranking_audit)
    stable_write_parquet(staged['score_blind_top5'], score_blind_top5)
    stable_write_csv(staged['condition_summary'], condition_summary)
    stable_write_csv(staged['input_inventory'], input_inventory)

    staged_hashes = OrderedDict(
        (key, sha256_file(staged[key]))
        for key in ('ranking_audit', 'score_blind_top5', 'condition_summary', 'input_inventory')
    )

    execution_report = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'notebook': NOTEBOOK_NAME,
        'project_root': str(ROOT),
        'authorization': {
            'cell_7c1_authorization_path': str(CELL_7C1['authorization']['path']),
            'cell_7c1_authorization_sha256': CELL_7C1['authorization']['sha256'],
            'authorization_decision': EXPECTED_CELL_7C1_AUTHORIZATION_DECISION,
            'authorized_cell': '7C2',
        },
        'frozen_design': {
            'questions': EXPECTED_QUESTIONS,
            'conditions': CONDITION_ORDER,
            'aliases': expected_alias_map,
            'semantic_candidate_pool_k': 20,
            'final_context_k': 5,
            'semantic_weight': 0.75,
            'quality_weight': 0.25,
            'rrf_constant': 60,
            'quality_rank_direction': 'higher quality signal receives better rank',
            'quality_rank_tie_breaker': ['rcv_accession ascending', 'packet_id ascending'],
            'final_rrf_tie_breaker': [
                'semantic_rank ascending',
                'quality_rank ascending',
                'rcv_accession ascending',
                'packet_id ascending',
            ],
            'hard_exclusion': False,
            'random_quality_seed': FROZEN_RANDOM_SEED,
            'random_quality_payload': '20260722|{question_id}|{rcv_accession}',
            'random_quality_integer_source': 'first 16 hexadecimal characters interpreted as unsigned 64-bit integer',
            'random_quality_scale': 'integer / (2^64 - 1)',
        },
        'execution_accounting': {
            'cell_7a3_score_rows_loaded': int(len(scores)),
            'frozen_top20_candidate_rows_loaded': int(len(top20)),
            'candidate_score_join_rows': int(len(joined)),
            'condition_specific_audit_rows': int(len(ranking_audit)),
            'score_blind_top5_rows': int(len(score_blind_top5)),
            'questions': int(ranking_audit['question_id'].nunique()),
            'conditions': int(ranking_audit['condition_id'].nunique()),
            'candidate_rows_per_question_condition': 20,
            'final_rows_per_question_condition': 5,
        },
        'scientific_operations': {
            'cell_7a3_scores_loaded': True,
            'quality_ranks_constructed_B_to_F': True,
            'quality_rank_constructed_A': False,
            'rrf_applied_B_to_F': True,
            'rrf_applied_A': False,
            'final_top5_materialized': True,
            'new_embeddings_generated': False,
            'new_semantic_retrieval_executed': False,
            'hard_evidence_exclusion_applied': False,
            'model_fitted_or_refitted': False,
            'threshold_or_weight_optimized': False,
            'prompts_materialized': False,
            'llm_called': False,
            'answer_keys_loaded_or_inspected': False,
            'adjudication_performed': False,
            'rag_or_retrieval_metrics_calculated': False,
        },
        'score_blind_downstream_boundary': {
            'later_prompt_stage_must_use': str(OUTPUTS['score_blind_top5']),
            'later_prompt_stage_must_not_use': str(OUTPUTS['ranking_audit']),
            'score_blind_top5_columns': list(score_blind_top5.columns),
            'scores_or_condition_identity_exposed_in_score_blind_top5': False,
        },
        'output_artifacts': {
            key: {
                'path': str(OUTPUTS[key]),
                'sha256': staged_hashes[key],
            }
            for key in ('ranking_audit', 'score_blind_top5', 'condition_summary', 'input_inventory')
        },
        'terminal_decision': terminal_decision,
        'next_authorized_cell': None,
        'next_required_action': (
            'Freeze a separate fail-closed authorization before any prompt materialization, '
            'LLM generation, answer-key inspection, adjudication, or RAG evaluation.'
        ),
    }
    stable_write_json(staged['execution_report'], execution_report)
    staged_hashes['execution_report'] = sha256_file(staged['execution_report'])

    qc_payload = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'prewrite_checks': {name: bool(value) for name, value in prewrite_checks.items()},
        'configuration_checks': {name: bool(value) for name, value in configuration_checks.items()},
        'passed_checks': int(len(prewrite_checks) + len(configuration_checks)),
        'failed_checks': 0,
        'total_checks': int(len(prewrite_checks) + len(configuration_checks)),
        'candidate_pool_rows': int(len(top20)),
        'audit_rows': int(len(ranking_audit)),
        'score_blind_top5_rows': int(len(score_blind_top5)),
        'leaking_score_blind_columns': leaking_columns,
        'terminal_decision': terminal_decision,
    }
    stable_write_json(staged['qc'], qc_payload)
    staged_hashes['qc'] = sha256_file(staged['qc'])

    manifest_payload = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'notebook': NOTEBOOK_NAME,
        'project_root': str(ROOT),
        'authorization_lineage': {
            'cell_7c1_authorization_sha256': CELL_7C1['authorization']['sha256'],
            'cell_7c1_manifest_sha256': CELL_7C1['manifest']['sha256'],
            'authorization_decision': EXPECTED_CELL_7C1_AUTHORIZATION_DECISION,
        },
        'upstream_lineage': {
            'cell_7c0_top20_sha256': CELL_7C0['semantic_top20_pool']['sha256'],
            'cell_7c0_manifest_sha256': CELL_7C0['manifest']['sha256'],
            'cell_7a3_score_table_sha256': EXPECTED_CELL_7A3_SCORE_SHA256,
            'cell_7a3_manifest_sha256': EXPECTED_CELL_7A3_MANIFEST_SHA256,
            'cell_7b4_embedding_config_sha256': EXPECTED_EMBEDDING_RETRIEVAL_CONFIG_SHA256,
            'cell_7b4_quality_config_sha256': EXPECTED_QUALITY_CONFIG_SHA256,
            'cell_7b4_condition_aliases_sha256': EXPECTED_CONDITION_ALIASES_SHA256,
            'cell_7b4_manifest_sha256': EXPECTED_CELL_7B4_MANIFEST_SHA256,
        },
        'frozen_output_artifacts': {
            key: {
                'path': str(OUTPUTS[key]),
                'sha256': staged_hashes[key],
            }
            for key in (
                'ranking_audit',
                'score_blind_top5',
                'condition_summary',
                'input_inventory',
                'execution_report',
                'qc',
            )
        },
        'scientific_boundary': {
            'same_frozen_top20_preserved_for_all_conditions': True,
            'cell_7a3_scores_loaded': True,
            'quality_reranking_executed': True,
            'fixed_rrf_executed': True,
            'final_top5_materialized': True,
            'score_blind_top5_materialized': True,
            'new_semantic_retrieval_executed': False,
            'hard_evidence_exclusion_applied': False,
            'prompts_materialized': False,
            'llm_called': False,
            'answer_keys_loaded_or_inspected': False,
            'adjudication_performed': False,
            'rag_or_retrieval_metrics_calculated': False,
        },
        'terminal_decision': terminal_decision,
        'next_authorized_cell': None,
        'next_required_action': (
            'Separate fail-closed authorization is required before prompt materialization or LLM execution.'
        ),
    }
    stable_write_json(staged['manifest'], manifest_payload)

    # Verify the complete staged package before copying anything to the frozen Drive locations.
    for key, path in staged.items():
        if not path.exists() or path.stat().st_size == 0:
            raise AssertionError(f'Staged Cell 7C2 artifact is missing or empty: {key}')

    staged_audit = pd.read_parquet(staged['ranking_audit'])
    staged_top5 = pd.read_parquet(staged['score_blind_top5'])
    staged_summary = pd.read_csv(staged['condition_summary'])
    staged_qc = load_json(staged['qc'])
    staged_manifest = load_json(staged['manifest'])

    staged_readback_checks = OrderedDict([
        ('staged_audit_9600', len(staged_audit) == EXPECTED_AUDIT_ROWS),
        ('staged_top5_2400', len(staged_top5) == EXPECTED_FINAL_TOP5_ROWS),
        ('staged_top5_exact_score_blind_columns',
         list(staged_top5.columns) == list(score_blind_top5.columns)),
        ('staged_summary_six_conditions', len(staged_summary) == 6),
        ('staged_qc_zero_failures', int(staged_qc['failed_checks']) == 0),
        ('staged_manifest_decision_exact', staged_manifest['terminal_decision'] == terminal_decision),
        ('staged_manifest_next_cell_none', staged_manifest['next_authorized_cell'] is None),
    ])
    failed_staged = [name for name, passed in staged_readback_checks.items() if not bool(passed)]
    if failed_staged:
        raise RuntimeError(
            'Staged Cell 7C2 package readback failed; nothing copied to frozen output paths:\n- '
            + '\n- '.join(failed_staged)
        )

    # Copy the fully verified package to final frozen paths.
    for key, final_path in OUTPUTS.items():
        final_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(staged[key], final_path)
        write_sidecar(final_path)

# Final readback from Drive.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing final Cell 7C2 output: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C2 output sidecar failed: {path}')

readback_audit = pd.read_parquet(OUTPUTS['ranking_audit'])
readback_top5 = pd.read_parquet(OUTPUTS['score_blind_top5'])
readback_summary = pd.read_csv(OUTPUTS['condition_summary'])
readback_qc = load_json(OUTPUTS['qc'])
readback_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('final_audit_rows_9600', len(readback_audit) == EXPECTED_AUDIT_ROWS),
    ('final_top5_rows_2400', len(readback_top5) == EXPECTED_FINAL_TOP5_ROWS),
    ('final_top5_exact_score_blind_columns',
     list(readback_top5.columns) == [
         'question_id', 'blinded_alias', 'context_position',
         'corpus_row_index', 'packet_id', 'rcv_accession'
     ]),
    ('final_top5_5_per_question_alias',
     readback_top5.groupby(['question_id', 'blinded_alias']).size().eq(5).all()),
    ('final_summary_six_conditions', len(readback_summary) == 6),
    ('final_qc_zero_failures', int(readback_qc['failed_checks']) == 0),
    ('final_manifest_decision_exact', readback_manifest['terminal_decision'] == terminal_decision),
    ('final_manifest_next_authorized_cell_none', readback_manifest['next_authorized_cell'] is None),
    ('all_output_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback_checks = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback_checks:
    raise RuntimeError(
        'Final Cell 7C2 frozen-package readback failed:\n- '
        + '\n- '.join(failed_readback_checks)
    )

# Reconfirm upstream immutability.
immutable_hashes_after = OrderedDict(
    (key, sha256_file(path)) for key, path in IMMUTABLE_SOURCE_PATHS.items()
)
if immutable_hashes_after != immutable_hashes_before:
    changed = [
        key for key in immutable_hashes_before
        if immutable_hashes_before[key] != immutable_hashes_after[key]
    ]
    raise AssertionError(
        'One or more frozen upstream artifacts changed during Cell 7C2:\n- '
        + '\n- '.join(changed)
    )

total_checks = (
    len(configuration_checks)
    + len(prewrite_checks)
    + len(staged_readback_checks)
    + len(readback_checks)
)
passed_checks = total_checks

separator = '=' * 156
print('\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C2')
print('FROZEN SIX-CONDITION QUALITY RERANKING, FIXED RRF, AND FINAL TOP-5 MATERIALIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\nUPSTREAM AUTHORIZATION AND INPUT REVERIFICATION')
print(f'Cell 7C1 manifest SHA-256                     : {sha256_file(CELL_7C1["manifest"]["path"])}')
print('Cell 7C1 terminal PASS verified               : YES')
print('Cell 7C2 authorization verified               : YES')
print(f'Cell 7C0 top-20 SHA-256                       : {sha256_file(CELL_7C0["semantic_top20_pool"]["path"])}')
print(f'Cell 7A3 score-table SHA-256                   : {sha256_file(CELL_7A3_SCORE_TABLE)}')
print(f'Cell 7B4 quality-config SHA-256                : {sha256_file(QUALITY_CONFIG)}')
print(f'Frozen semantic candidate rows                : {len(top20):,}')
print(f'Cell 7A3 score rows loaded                     : {len(scores):,}')
print(f'Candidate-score joins                         : {len(joined):,}/{len(top20):,}')

print('\nFROZEN SIX-CONDITION EXECUTION')
print('Conditions                                    : 6')
print('A / ARM-MICA                                  : semantic-only; semantic ranks 1-5')
print('B / ARM-ORBIT                                 : review/conflict-aware quality + fixed RRF')
print('C / ARM-KITE                                  : combined-metadata quality + fixed RRF')
print('D / ARM-PULSE                                 : Full-GES P(stable) quality + fixed RRF')
print('E / ARM-LARCH                                 : no-star-GES P(stable) quality + fixed RRF')
print('F / ARM-NOVA                                  : deterministic SHA-256 random quality + fixed RRF')
print('Semantic candidate pool                       : exact frozen top-20; unchanged')
print('Quality ranking                               : higher quality = better rank')
print('Soft rank fusion                              : 0.75 semantic + 0.25 quality; RRF constant 60')
print('Final context                                 : top-5 per question-condition')
print(f'Condition-specific audit rows                 : {len(readback_audit):,}')
print(f'Score-blind final top-5 rows                  : {len(readback_top5):,}')
print('Hard evidence exclusion                       : NO')

print('\nCELL 7C2 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\nSCIENTIFIC OPERATIONS IN CELL 7C2')
print('Cell 7A3 row-level scores loaded              : YES')
print('Quality ranks constructed B-F                : YES')
print('RRF applied B-F                              : YES')
print('Final top-5 contexts materialized            : YES')
print('New embeddings generated                     : NO')
print('New semantic retrieval executed              : NO')
print('Hard evidence exclusion                      : NO')
print('Prompts materialized                         : NO')
print('LLM called                                   : NO')
print('Answer-key outcomes inspected                : NO')
print('Adjudication or RAG metrics                  : NO')

print('\nNEXT AUTHORIZATION BOUNDARY')
print('Next execution cell                           : NOT AUTHORIZED')
print('Required next action                          : separate fail-closed authorization before')
print('                                                 prompt materialization or LLM generation')
print('Later prompt input                            : score-blind Cell 7C2 top-5 selection ONLY')
print('Score-bearing reranking audit to LLM          : PROHIBITED')

print(f'\nFINAL DECISION                                : {terminal_decision}')
print(separator)


EXPERIMENT 2 — STAGE 7C — CELL 7C2
FROZEN SIX-CONDITION QUALITY RERANKING, FIXED RRF, AND FINAL TOP-5 MATERIALIZATION
Notebook                                      : 09_GES_Aware_Genomic_RAG_Cell_7C2_Quality_Reranking_and_Top5_Execution.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION AND INPUT REVERIFICATION
Cell 7C1 manifest SHA-256                     : f5a31f64e47e60087365ee58e75880354c48f76ef7fc941e5840aa1ec7360466
Cell 7C1 terminal PASS verified               : YES
Cell 7C2 authorization verified               : YES
Cell 7C0 top-20 SHA-256                       : be7f7cfc369952fc905cde5fb843e126b8e36d86f9bc625be853e9489d5327cb
Cell 7A3 score-table SHA-256                   : e9b162c9add5aed34a4d68d8bf625251293a18c549945468e627a498843eb802
Cell 7B4 quality-config SHA-256                : 35c3871db6dd6bad9436e7202a67b064cc98beee06d19894e409d42f7ca006fb
Frozen semantic candidate rows                : 1,600
C